# DL · Lecture 14 — Decoder-only Transformer & Intro to LLMs

You've met attention and encoder–decoders; now **build a decoder-only Transformer from scratch** and train it char-level on **tiny-shakespeare** — the same architecture as GPT, just small. You'll see the **causal mask** that makes it autoregressive, **multi-head self-attention** and **positional encoding**, **next-token** training, and **autoregressive generation** with temperature / top-k / top-p. It generates Shakespeare-ish text by the end.

Follows the course rhythm: **setup/GPU-check → get the data → look at the data → build (🔧) → train (🔧) → generate → experiment → reflect**. Real `torch` — the dataset downloads at runtime, no `pip install`.

> **Run on a GPU:** Colab → **Runtime → Change runtime type → T4 GPU** (training is ~2–3 min on a T4; it works on CPU too, just slower — drop `MAX_STEPS`).

> 🎥 New to this? Watch StatQuest — [Decoder-Only Transformers (ChatGPT), clearly explained](https://www.youtube.com/watch?v=bQ5BoolX9Ag) — then build one here.

## 0. Setup & GPU check

In [ ]:
import math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU for the intended "
          "~2-3 min run (it still works on CPU — lower MAX_STEPS).")

# Colab-friendly model + training sizes. Scale up on a GPU for crisper text.
BLOCK_SIZE = 128      # context window (tokens the model attends over)
D_MODEL, N_HEADS, N_LAYERS, D_FF = 192, 6, 4, 768
BATCH, MAX_STEPS, LR = 64, 2000, 3e-4

## 1. From encoder–decoder to decoder-only

A **decoder-only** model is a *single* stack of identical blocks — no separate encoder. There's just a sequence of tokens and the model predicts the next one, so **conditioning on a prompt and generating a continuation are the same operation**. Each block is two sublayers wrapped with **residuals** and **LayerNorm**:

1. **Causal (masked) self-attention** — mixes information across positions.
2. **A position-wise feed-forward network** — a small MLP applied at each position.

We use **pre-norm** (LayerNorm *before* each sublayer, as GPT-2+ do): it leaves the residual path one unbroken sum from input to output, so gradients flow straight back and deep stacks train stably.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, attn_mask):
        h = self.norm1(x)                                    # pre-norm
        a, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)  # self-attention: Q=K=V=h
        x = x + a                                            # residual
        x = x + self.ff(self.norm2(x))                       # residual
        return x

print(DecoderBlock(D_MODEL, N_HEADS, D_FF))

## 2. The causal mask: only look at the past

To predict the next token, a position must **never** see tokens after it — else it'd read the answer. The **causal mask** sets future cells to `-inf` *before* the softmax, so their attention weight becomes 0. This is exactly what makes the model **autoregressive**.

In [ ]:
def causal_mask(seq_len, device="cpu"):
    # -inf above the diagonal = "not allowed to attend"
    return torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=device), diagonal=1)

print(causal_mask(4))
# each row t may attend only to columns 0..t (lower triangle incl. diagonal)

## 3. Get the data — and look at it

We train **char-level** on **tiny-shakespeare** (~1 MB of the plays as one text file). The "tokenizer" is the simplest possible: **each character is a token**. A real LLM uses sub-word tokens, but char-level keeps the whole pipeline visible with no dependencies.

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(URL).read().decode("utf-8")
print(f"{len(text):,} characters")
print(text[:250])

chars = sorted(set(text))
VOCAB = len(chars)
stoi = {c: i for i, c in enumerate(chars)}      # char -> id
itos = {i: c for i, c in enumerate(chars)}      # id -> char
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
print(f"\nvocab = {VOCAB} characters:", "".join(chars).replace("\n", "\\n"))

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print("train/val tokens:", len(train_data), len(val_data))

## 4. Tokens, embeddings, and the full model (🔧)

The pipeline: **tokenize** (done) → **embed** ids to vectors and **add positional information** (attention is permutation-invariant, so order must be injected) → **N decoder blocks** → an **output head** producing **logits**, one score per vocab entry. We use a *learned* position embedding (a trainable table indexed by position — modern LLMs often use rotary/RoPE, but the idea is the same).

In [ ]:
class GPTMini(nn.Module):
    def __init__(self, vocab, d_model, n_heads, n_layers, d_ff, block_size):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab, d_model)          # id -> vector
        self.pos_emb = nn.Embedding(block_size, d_model)     # position -> vector (learned)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab)                # final vectors -> logits

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)            # embed + position
        mask = causal_mask(T, idx.device)
        for blk in self.blocks:
            x = blk(x, mask)                                 # mix context, causally
        return self.head(self.norm(x))                       # (B, T, vocab) logits

model = GPTMini(VOCAB, D_MODEL, N_HEADS, N_LAYERS, D_FF, BLOCK_SIZE).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 5. Train: next-token prediction (🔧)

The objective is strikingly simple — at **every** position, predict the next token — and the label is just the next character already in the text. No human labelling: this is **self-supervised**. The loss is **cross-entropy** between the predicted distribution and the true next token, and thanks to the causal mask a **single forward pass** scores the prediction at every position at once.

In [ ]:
def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK_SIZE - 1, (BATCH,))
    x = torch.stack([d[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i + 1:i + BLOCK_SIZE + 1] for i in ix])   # targets = inputs shifted by 1
    return x.to(device), y.to(device)

@torch.no_grad()
def val_loss():
    model.eval()
    losses = [F.cross_entropy(model(xb).reshape(-1, VOCAB), yb.reshape(-1)).item()
              for xb, yb in (get_batch("val") for _ in range(20))]
    model.train()
    return sum(losses) / len(losses)

opt = torch.optim.AdamW(model.parameters(), lr=LR)
model.train()
for step in range(1, MAX_STEPS + 1):
    xb, yb = get_batch("train")
    logits = model(xb)                                           # (B, T, vocab)
    # get_batch already shifted y by one, so EVERY position is a valid next-token target -> no slicing.
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), yb.reshape(-1))   # predict next token everywhere
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 250 == 0 or step == 1:
        print(f"step {step:4d}/{MAX_STEPS}  train {loss.item():.3f}  val {val_loss():.3f}")

## 6. Autoregressive generation

To *generate*, run the loop the decoder was trained for: feed the sequence in, take the logits at the **last** position, softmax → next-token probs, **sample** one, **append**, repeat. **Temperature** rescales logits before softmax (`<1` sharpens, `>1` flattens); **top-k** keeps the `k` best tokens; **top-p (nucleus)** keeps the smallest set whose cumulative probability reaches `p`. We crop the context to the last `block_size` tokens each step (the context window).

In [ ]:
@torch.no_grad()
def generate(prompt="\n", max_new=400, temperature=0.8, top_k=None, top_p=None):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new):
        logits = model(idx[:, -model.block_size:])[:, -1, :] / temperature   # last position, temp
        probs = F.softmax(logits, dim=-1)
        if top_k:                                     # keep k highest, renormalise
            v, i = torch.topk(probs, min(top_k, probs.size(-1)))
            probs = torch.zeros_like(probs).scatter_(1, i, v)
        if top_p:                                     # nucleus: smallest set reaching p (batch size 1 here)
            s, i = torch.sort(probs, descending=True)
            cut = (torch.cumsum(s, -1) >= top_p).float().argmax(-1)
            s[0, cut + 1:] = 0
            probs = torch.zeros_like(probs).scatter_(1, i, s)
        probs = probs / probs.sum(-1, keepdim=True)
        nxt = torch.multinomial(probs, 1)             # sample one token
        idx = torch.cat([idx, nxt], dim=1)            # append, then repeat
    return decode(idx[0].tolist())

print(generate("ROMEO:", max_new=400, temperature=0.8, top_k=40))

> **Going deeper — the KV-cache.** Our `generate` re-feeds the *entire* sequence every step, so generating token `t` re-encodes all `t-1` earlier ones — quadratic work as the text grows. Real implementations **cache** the keys and values of every token already processed: each step computes Q/K/V for only the **one** new token, attends it against the cached K/V, and appends its own — dropping per-step cost from quadratic to **linear** (at the price of memory that grows with context length). It's the single most important fact about how decoder-only models are *served*; we skip it here for clarity.

### Greedy vs. sampling
Same model, same probabilities — only the **selection rule** differs. **Greedy** (`top_k=1`, i.e. argmax) always takes the single most likely token, so a prompt gives the *same* output every time (and tends to repeat). **Sampling** varies run to run and reads more naturally. (Don't reach for `temperature=0` to get greedy — that divides the logits by zero; use `top_k=1`.)

In [ ]:
print("=== greedy (top_k=1, deterministic) ===")
print(generate("ROMEO:", max_new=200, top_k=1))
print("\n=== sampling (varies, temp=1.0) ===")
print(generate("ROMEO:", max_new=200, temperature=1.0, top_k=40))

### The temperature / softmax worked example (always runs)
The lecture's §5 example: logits `[3.0, 2.0, 1.5]` → softmax `[0.63, 0.23, 0.14]`. See how temperature reshapes it — low `T` sharpens toward the top token, high `T` flattens toward uniform:

In [ ]:
z = torch.tensor([3.0, 2.0, 1.5])
for T in [0.5, 1.0, 2.0]:
    p = F.softmax(z / T, dim=-1)
    print(f"T={T}:  {[round(x, 2) for x in p.tolist()]}")
# T=1.0 -> [0.63, 0.23, 0.14]; T<1 sharpens; T>1 flattens

### Top-k vs top-p, by hand (always runs)
For probs `[0.6, 0.3, 0.07, 0.03]`: **top-k=2** keeps the first two → `[0.67, 0.33]`; **top-p=0.9** also stops after two (`0.6 + 0.3 = 0.9`), dropping the tail. top-k keeps a *fixed* count; top-p's cut-off is *adaptive*.

In [ ]:
probs = torch.tensor([0.6, 0.3, 0.07, 0.03])
def tk(p, k):
    v, i = torch.topk(p, k); out = torch.zeros_like(p); out[i] = v; return (out / out.sum()).tolist()
def tp(p, thr):
    s, i = torch.sort(p, descending=True); cut = int((torch.cumsum(s, 0) >= thr).float().argmax()) + 1
    out = torch.zeros_like(p); out[i[:cut]] = p[i[:cut]]; return (out / out.sum()).tolist()
print("top-k=2:", [round(x, 2) for x in tk(probs, 2)], " (expect .67,.33,0,0)")
print("top-p=.9:", [round(x, 2) for x in tp(probs, 0.9)], " (expect .67,.33,0,0)")

## 7. Peek inside: what a head attends to

Multi-head attention runs `h` attentions in parallel; each **head** learns a different relevance pattern. Pull the attention weights for one head on a short prompt and look at the **lower-triangular** structure — every row sums to 1 over *past* positions only (the causal mask in action):

In [ ]:
@torch.no_grad()
def attention_row(prompt="ROMEO:"):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    T = idx.size(1)
    x = model.tok_emb(idx) + model.pos_emb(torch.arange(T, device=device))
    blk = model.blocks[0]
    h = blk.norm1(x)
    _, w = blk.attn(h, h, h, attn_mask=causal_mask(T, device), need_weights=True, average_attn_weights=True)
    return w[0]                                       # (T, T) averaged over heads

w = attention_row("ROMEO:")
print("attention weights (row = query position, col = attended position):")
for r, ch in enumerate("ROMEO:"):
    print(f"  {ch!r}: " + " ".join(f"{x:.2f}" for x in w[r].tolist()))
print("Upper triangle is ~0 (future is masked); each row sums to 1 over the past.")

## 8. Practice

**P1 — decoding knobs.** Re-run §6 `generate(...)` with `temperature` in {0.3, 0.8, 1.3} and compare coherence vs variety. Then try `top_p=0.9` instead of `top_k`. Which feels most natural?

**P2 — the causal mask is load-bearing.** In `DecoderBlock.forward`, pass `attn_mask=None` (delete the mask), retrain for a few hundred steps, and watch the **val loss collapse toward ~0** while generation turns to garbage — the model cheated by reading the next token. Put the mask back.

**P3 — context window.** Lower `BLOCK_SIZE` to 16 and retrain. Long-range structure (line breaks, speaker names) degrades — the model can't attend past 16 chars. Why does raising it cost more? *(attention is quadratic in sequence length.)*

**P4 (paper).** Why does next-token prediction need **no human labels**, and name one capability that *emerges* from it at scale without being trained for directly. **P5:** this is the bridge to the *Building LLM-Powered Applications* course — the class model is exactly this (a pretrained, instruction-tuned decoder-only Transformer), just billions of parameters instead of ~2 million.

**P6 (reflect).** Your LLM is a *generative* model — it models `p(next token | context)` and samples. The lecture's §8 places it alongside the other three deep-generative families (VAE, GAN, diffusion); skim that section and note which family targets images rather than text.

## Key terms
- **Decoder-only Transformer** — one stack of causal-self-attention + feed-forward blocks; reads context, predicts the next token.
- **Causal mask** — forces each position to attend only to earlier positions (makes it autoregressive).
- **Self-attention** — Q, K, V are learned projections of the *same* sequence.
- **Multi-head attention** — `h` attentions in parallel over split chunks; each head a different relevance pattern.
- **Positional encoding** — position injected into embeddings, since attention is permutation-invariant.
- **Next-token prediction** — self-supervised objective; the label is the next token, free from the data.
- **Temperature / top-k / top-p** — the knobs that trade determinism for diversity at generation time.